# FingerBall Stage 2J-1B | Finger-specific QC & interactive 3D comparison

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atstuyuki/FingerBall-Analysis/blob/main/notebooks/FingerBall_Stage2J1B_Detailed_QC_3D_Viewer.ipynb)

**目的：** Stage 2J-1で成功した推論結果を再利用し、リリース前後の**示指・中指（MCP/PIP/DIP/TIP）**の2D誤差を詳しく検証する。MediaPipe / WiLoR-miniの**モデル別推定3D**は、360°回転＋フレーム操作可能なHTMLビューアで比較する。

- **推論は再実行しません**（WiLoRの環境構築も不要）。
- Stage 2Hで目視確認済みのXYは一切変更しません。
- 参照データを使ったROI補助条件は独立した検出性能ではありません。
- MediaPipeとWiLoRの推定3Dは、座標系とスケールが異なります。ビューアでは手首原点・手掌幅で**表示目的に正規化**します。**真の3D精度比較ではありません**。
- 無検出や遮蔽は欠損のまま残します。勝手な時系列補間はしません。

**Stage 2J-2 SOMA統合前のQCゲート**を作るNotebookです。

## 0. Google Driveとライブラリ

In [ ]:
%pip -q install "pandas>=2.0" "numpy>=1.24" "plotly>=5.18" "matplotlib>=3.7" "opencv-python-headless>=4.8"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from IPython.display import display,HTML
import pandas as pd,numpy as np,matplotlib.pyplot as plt
import cv2,json,hashlib,html
import plotly.graph_objects as go
import plotly.io as pio

## 1. 入力・検証・データ保全

Stage 2J-1で出力された共通CSVを指定します。モデル別の再推論は行いません。

In [ ]:
PROJECT=Path('/content/drive/MyDrive/母指回内トラッキング')
VIDEO_STEM='7A5812501DC5F7120DF842D06EE1B74C695459F1'
INPUT_CSV=PROJECT/'FingerBall_v2_stage2j1_results'/VIDEO_STEM/'stage2j1_common_21_landmarks_with_provenance.csv'
SOURCE_VIDEO=PROJECT/(VIDEO_STEM+'.mp4')  # optional
OUT=PROJECT/'FingerBall_v2_stage2j1b_results'/VIDEO_STEM
FRAMES=list(range(185,199))
DETAIL_FRAMES=list(range(188,194))
RELEASE_REFERENCE=190
METHODS=['mediapipe_full','mediapipe_manual_roi','wilor_full_detector','wilor_manual_bbox']
FINGER_IDS={'Index':(5,6,7,8),'Middle':(9,10,11,12)}
FINGER_JOINTS=('MCP','PIP','DIP','TIP')
if not INPUT_CSV.is_file():raise FileNotFoundError('Stage 2J-1 CSVが見つかりません: '+str(INPUT_CSV))
OUT.mkdir(parents=True,exist_ok=True)
source_digest=hashlib.sha256(INPUT_CSV.read_bytes()).hexdigest()
data=pd.read_csv(INPUT_CSV)
required={'frame','method','landmark_id','stage2h_x_px','stage2h_y_px',
          'x_px','y_px','model3d_x','model3d_y','model3d_z',
          'status_stage2h','detected','assisted_by_stage2h_roi'}
assert required.issubset(data.columns),required-set(data.columns)
if data.duplicated(['frame','method','landmark_id']).any():raise ValueError('CSV duplicate frame/method/landmark')
subset=data[data.frame.isin(FRAMES)].copy()
for f in FRAMES:
    for m in METHODS+['stage2h_reference']:
        if subset[(subset.frame==f)&(subset.method==m)].landmark_id.nunique()!=21:
            raise ValueError(f'21 points missing: frame={f} method={m}')
data=subset
print('Input SHA256:',source_digest,'rows:',len(data))
print('Detected frames:',data[data.method.isin(METHODS)].groupby('method').detected.sum()/21)

## 2. 指・関節ごとの誤差集計（手動点のみも別集計）

各ランドマークの**XY Euclidean error**を再計算。欠損と遮蔽は除外。全フレームで検出できていないモデルは欠測の多さも必ず表示します。

In [ ]:
eligible=data[data.method.isin(METHODS)].copy()
eligible['ref_valid']=eligible[['stage2h_x_px','stage2h_y_px']].notna().all(axis=1)
eligible['pred_valid']=eligible[['x_px','y_px']].notna().all(axis=1)&eligible.detected.fillna(False)
eligible['evaluated']=eligible.ref_valid&eligible.pred_valid&(~eligible.status_stage2h.isin(['occluded','missing']))
eligible['error_px']=np.where(eligible.evaluated,np.hypot(
    eligible.x_px-eligible.stage2h_x_px,
    eligible.y_px-eligible.stage2h_y_px),np.nan)
joint_lookup={idx:(finger,joint) for finger,ids in FINGER_IDS.items()
              for idx,joint in zip(ids,FINGER_JOINTS)}
eligible['finger']=eligible.landmark_id.map(lambda x:joint_lookup.get(x,('Other','Other'))[0])
eligible['joint']=eligible.landmark_id.map(lambda x:joint_lookup.get(x,('Other','Other'))[1])
finger=eligible[eligible.finger.isin(FINGER_IDS)].copy()
finger.to_csv(OUT/'stage2j1b_jointwise_long.csv',index=False)
def rmse(v):
    v=np.asarray(pd.Series(v).dropna(),float)
    return np.sqrt(np.mean(v*v)) if len(v) else np.nan
agg=(finger.groupby(['method','finger','joint'],as_index=False)
    .agg(n_evaluable=('error_px','count'),
         n_observation_rows=('error_px','size'),
         rmse_px=('error_px',rmse),
         median_px=('error_px','median'),
         max_error_px=('error_px','max')))
manual=(finger[finger.status_stage2h.eq('manual_visible')]
    .groupby(['method','finger','joint'],as_index=False)
    .agg(n_manual=('error_px','count'),manual_rmse_px=('error_px',rmse)))
agg=agg.merge(manual,on=['method','finger','joint'],how='left')
agg.to_csv(OUT/'stage2j1b_jointwise_summary.csv',index=False)
frameagg=(finger.groupby(['method','frame','finger'],as_index=False)
    .agg(n_evaluable=('error_px','count'),rmse_px=('error_px',rmse)))
frameagg.to_csv(OUT/'stage2j1b_per_frame_finger_rmse.csv',index=False)
display(agg.round(2))
print('Reminder: small RMSE on only a few detected landmarks does not imply good detection rate')

## 3. 示指・中指の時間変化と関節別誤差ヒートマップ

In [ ]:
for fname in FINGER_IDS:
    t=frameagg[frameagg.finger==fname]
    fig,ax=plt.subplots(figsize=(11,4.5))
    for method in METHODS:
        z=t[t.method==method]
        ax.plot(z.frame,z.rmse_px,marker='o',label=method)
    ax.axvline(RELEASE_REFERENCE,color='black',linestyle='--',alpha=.6)
    ax.set(title=f'{fname} hand landmarks 2D error to Stage 2H (detected only)',
           xlabel='Original video frame',ylabel='RMSE pixels')
    ax.legend(fontsize=8);ax.grid(alpha=.2);fig.tight_layout()
    fig.savefig(OUT/f'stage2j1b_{fname.lower()}_rmse_timeseries.png',dpi=160);plt.show()
for method in METHODS:
    h=agg[agg.method==method].copy()
    mat=h.pivot(index='finger',columns='joint',values='rmse_px').reindex(
        index=['Index','Middle'],columns=list(FINGER_JOINTS))
    fig,ax=plt.subplots(figsize=(6.5,2.8))
    im=ax.imshow(mat.values,aspect='auto',cmap='viridis',vmin=0)
    ax.set_xticks(np.arange(4),mat.columns)
    ax.set_yticks(np.arange(2),mat.index)
    ax.set_title(method+' | 2D RMSE (pixels)')
    for i in range(2):
        for j in range(4):
            v=mat.values[i,j]
            ax.text(j,i,f'{v:.1f}' if np.isfinite(v) else 'NA',ha='center',
                    va='center',color='white',fontsize=10)
    fig.colorbar(im,ax=ax,label='RMSE px')
    fig.tight_layout()
    fig.savefig(OUT/f'stage2j1b_joint_heatmap_{method}.png',dpi=160);plt.show()